# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **How many potholes were filled last month in Pittsburgh?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-11-13 22:21:50 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-11-13T22:21:50.921963")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'pothole repair requests Pittsburgh':

1. **Pittsburgh 311 Data** (43.9% match)
   Resource: 311 Data
   Resource ID: `5202679a-d243-402e-b82a-63189995a942`
   Dataset ID: `pittsburgh-311-data`
   Format: CSV | Rows: 934,488 | Cols: 26
   Tags: service_request, public_services, city_services, operations, infrastructure, issues, maintenance, reporting, compliance, utilities
   This dataset contains a comprehensive collection of case information, likely related to city services or public requests. The data spans a significant period, with creation and
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'pothole repair requests Pittsburgh', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for '311 service requests potholes':

1. **Pittsburgh 311 Data** (42.5% match)
   Resource: 311 Data
   Resource ID: `5202679a-d243-402e-b82a-63189995a942`
   Dataset ID: `pittsburgh-311-data`
   Format: CSV | Rows: 934,488 | Cols: 26
   Tags: service_request, public_services, city_services, operations, infrastructure, issues, maintenance, reporting, compliance, utilities
   This dataset contains a comprehensive collection of case information, likely related to city services or public requests. The data spans a significant period, with creation and modi
```


In [ ]:
# Step 2: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": '311 service requests potholes', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 982,818
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject         created_date_et        created_date_utc   last_modified_da
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query

**SQL:**
```sql
SELECT DISTINCT subject, subject_code FROM "5202679a-d243-402e-b82a-63189995a942" WHERE subject ILIKE '%pothole%'
```

**Result preview:**
```
SQL: SELECT DISTINCT subject, subject_code FROM "5202679a-d243-402e-b82a-63189995a942" WHERE subject ILIKE '%pothole%' LIMIT 10000
Rows: 1
Columns: subject, subject_code

 subject subject_code
Potholes         PTHL
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT DISTINCT subject, subject_code FROM "5202679a-d243-402e-b82a-63189995a942" WHERE subject ILIKE \'%pothole%\''

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT MAX(created_date_et) as max_date FROM "5202679a-d243-402e-b82a-63189995a942"
```

**Result preview:**
```
SQL: SELECT MAX(created_date_et) as max_date FROM "5202679a-d243-402e-b82a-63189995a942" LIMIT 10000
Rows: 1
Columns: max_date

           max_date
2026-11-13T12:59:26
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT MAX(created_date_et) as max_date FROM "5202679a-d243-402e-b82a-63189995a942"'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT to_char(created_date_et::timestamp, 'YYYY-MM') as month, status, COUNT(*) as cnt
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE subject_code = 'PTHL'
GROUP BY month, status
ORDER BY month DESC
LIMIT 30
```

**Result preview:**
```
SQL: SELECT to_char(created_date_et::timestamp, 'YYYY-MM') as month, status, COUNT(*) as cnt
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE subject_code = 'PTHL'
GROUP BY month, status
ORDER BY month DESC
LIMIT 30
Rows: 30
Columns: month, status, cnt

  month             status  cnt
2026-11             Closed   24
2026-11               Open   25
2026-11   Open (Duplicate)    1
2026-10             Closed  113
2026-10 Closed (Duplicate)   10
2026-10               Open   28
2026-10   Open (Duplicate)    7
2026-09             Closed  186
2026-09 Closed (Duplicate)   13
2026-09        In Progres
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT to_char(created_date_et::timestamp, \'YYYY-MM\') as month, status, COUNT(*) as cnt\nFROM "5202679a-d243-402e-b82a-63189995a942"\nWHERE subject_code = \'PTHL\'\nGROUP BY month, status\nORDER BY month DESC\nLIMIT 30'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql
SELECT to_char(closed_date_et::timestamp, 'YYYY-MM') as month, status, COUNT(*) as cnt
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE subject_code = 'PTHL' AND closed_date_et IS NOT NULL
GROUP BY month, status
ORDER BY month DESC
LIMIT 20
```

**Result preview:**
```
SQL: SELECT to_char(closed_date_et::timestamp, 'YYYY-MM') as month, status, COUNT(*) as cnt
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE subject_code = 'PTHL' AND closed_date_et IS NOT NULL
GROUP BY month, status
ORDER BY month DESC
LIMIT 20
Rows: 20
Columns: month, status, cnt

  month             status  cnt
2026-11             Closed   88
2026-11 Closed (Duplicate)    8
2026-10             Closed  173
2026-10 Closed (Duplicate)   16
2026-09             Closed  290
2026-09 Closed (Duplicate)   16
2026-08             Closed  179
2026-08 Closed (Duplicate)   28
2026-07             Closed 
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT to_char(closed_date_et::timestamp, \'YYYY-MM\') as month, status, COUNT(*) as cnt\nFROM "5202679a-d243-402e-b82a-63189995a942"\nWHERE subject_code = \'PTHL\' AND closed_date_et IS NOT NULL\nGROUP BY month, status\nORDER BY month DESC\nLIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **Based on the **Pittsburgh 311 Data** dataset (WPRDC portal, resource ID `5202679a-d243-402e-b82a-63189995a942`), here's what I found for pothole-related requests (subject code **PTHL**):

**Note on "last month":** The most recent data in this dataset runs through **2026‑11‑13**, so the last *complete* month available is **October 2026**. (This appears to be a continuously-updated/synthetic dataset where the system clock reflects a future date — I'm using the most recent full month as "last month.")

**Potholes closed (i.e., filled/resolved) in October 2026:**
- By **closed date**: 173 pothole requests were closed in October 2026 (plus 16 marked "Closed (Duplicate)")
- By **created date** (i.e., reported in October and closed by now): 113 were marked Closed (plus 10 duplicates), with 28 still Open

**Best answer: ~173 pothole 311 requests were closed/filled in October 2026**, using the closed_date field, which most directly reflects when repair work was completed.

**Caveats:**
- "Closed" in 311 data indicates the service request was resolved/closed — this is used as a proxy for "filled," since the dataset doesn't have an explicit "pothole filled" field distinct from case closure.
- "Closed (Duplicate)" cases (16) were excluded from the 173 figure since those represent duplicate reports, not necessarily independent repairs.
- If you intended a specific calendar month (e.g., the actual current month when you're asking), let me know and I can re-run the query for that exact month.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-11-13

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-11-13 22:21:51
- **Query**: How many potholes were filled last month in Pittsburgh?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
